# 1、Chains的基本使用

Chain：链，用于将多个组件（提示模板、LLM模型、记忆、工具等）连接起来，形成可复用的工作流，完成复杂的任务。

## LCEL 及其基本构成

LangChain表达式语言（LCEL，LangChain Expression Language）是一种声明式方法，可以轻松地将多个组件链接成 AI 工作流。它通过Python原生操作符（如管道符| ）将组件连接成可执行流程，显著简化了AI应用的开发。

LCEL的基本构成：提示（Prompt）+ 模型（Model）+ 输出解析器（OutputParser）

在使用LCEL之前

In [4]:
# 引入依赖包
from langchain_core.output_parsers import JsonOutputParser,StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
import os
import dotenv

dotenv.load_dotenv()  #加载当前目录下的 .env 文件

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")


# 初始化语言模型
chat_model = ChatOpenAI(model="gpt-4o-mini")

joke_query = "告诉我一个笑话。"

# 定义Json或Str解析器
# parser = JsonOutputParser()
parser = StrOutputParser()


# 定义提示词模版
# 注意，提示词模板中需要部分格式化解析器的格式要求format_instructions
prompt_template = PromptTemplate(
    # template="回答用户的查询.\n满足的格式为{format_instructions}\n问题为{question}\n",
    # partial_variables={"format_instructions": parser.get_format_instructions()},
    template="给我讲一个关于{topic}话题的简短笑话"
)

#prompt = prompt_template.invoke(input={"question":joke_query})
prompt = prompt_template.invoke({"topic":"冰淇淋"})
response = chat_model.invoke(prompt)
print(response)

result = parser.invoke(response)
print(result)


content='为什么冰淇淋从来不吵架？\n\n因为它们总是“融洽”相处！' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 24, 'total_tokens': 51, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f97eff32c5', 'id': 'chatcmpl-D5qDK4bpJrovhgFjOvjtAdByIJGp9', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019c2d31-2844-7d60-b743-4364f19eeeac-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 24, 'output_tokens': 27, 'total_tokens': 51, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
为什么冰淇淋从来不吵架？

因为它们总是“融洽”相处！


情况2：使用chain：将提示模板、模型、解析器链接在一起。使用LCEL将不同的组件组合成一个单一的链条

In [7]:
# 引入依赖包
from langchain_core.output_parsers import JsonOutputParser,StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
import os
import dotenv

dotenv.load_dotenv()  #加载当前目录下的 .env 文件

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")


# 初始化语言模型
chat_model = ChatOpenAI(model="gpt-4o-mini")

joke_query = "告诉我一个笑话。"

# 定义Json或Str解析器
parser = JsonOutputParser()


# 定义提示词模版
# 注意，提示词模板中需要部分格式化解析器的格式要求format_instructions
prompt_template = PromptTemplate(
    template="回答用户的查询.\n满足的格式为{format_instructions}\n问题为{question}\n",
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

prompt = prompt_template.invoke(input={"question":joke_query})

chain =  prompt_template|chat_model|parser
out_put = chain.invoke(prompt)
print(out_put)


{'joke': '为什么鸡过马路？为了到另一边！'}


# 2、传统Chain的使用

## 2.1 基础链：LLMChain

LCEL之前，最基础也最常见的链类型是LLMChain。

这个链至少包括一个提示词模板（PromptTemplate），一个语言模型（LLM 或聊天模型）。

特点：

用于 单次问答，输入一个 Prompt，输出 LLM 的响应。

适合 无上下文 的简单任务（如翻译、摘要、分类等）。

无记忆：无法自动维护聊天历史

使用步骤：

1、配置任务链

chain = LLMChain(llm = llm, prompt 1 = prompt_template)

2、执行任务链

chain.invoke(...)

举例1

In [2]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os
import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 1、创建大模型实例
chat_model = ChatOpenAI(model="gpt-4o-mini")

# 2、原始字符串模板
template = "桌上有{number}个苹果，四个桃子和 3 本书，一共有几个水果?"
prompt = PromptTemplate.from_template(template)

# 3、创建LLMChain
# llm_chain = LLMChain(
#     llm=chat_model,
#     prompt=prompt
# )

llm_chain = prompt | chat_model | StrOutputParser()  #直接str化

# 4、调用LLMChain，返回结果
result = llm_chain.invoke({"number": 2})
print(result)

桌上有2个苹果和4个桃子，所以一共有水果的数量是：

2 + 4 = 6

因此，桌上一共有6个水果。


In [7]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os
import dotenv
from langchain_openai import ChatOpenAI
from langchain_classic.chains import LLMChain

dotenv.load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 1、创建大模型实例
chat_model = ChatOpenAI(model="gpt-4o-mini")

# 2、提供提示词模板
# BasePromptTemplate的典型子类：PromptTemplate、ChatPromptTemplate
prompt_template=PromptTemplate.from_template(
    template = "你是一个数据高手，帮我解决如下数学问题：{question}"
)

chain = LLMChain(llm=chat_model,prompt=prompt_template)
response = chain.invoke(input={"question":"1+2+3 = ?"})
print(response)

{'question': '1+2+3 = ?', 'text': '1 + 2 + 3 = 6。'}


举例2：verbose参数，使用使用ChatPromptTemplate。

prompt | llm 是 LangChain 1.0+ 的简易链式调用（基于 Runnable 接口），这种写法本身不支持 verbose 参数，如果需要查看运行过程，有两种替代方案：

1、用 stream() 方法替代 invoke()：流式输出模型的返回结果，间接查看运行过

2、封装为自定义 Chain：将简易链条封装为支持 verbose 的高级 Chain（略复杂，新手不推荐）

补充说明：调用方法除了invoke()外，还有run()、predict()、实例方法等，效果与invoke()相同，这里不再介绍。

使用ChatPromptTemplate，设置verbose参数

In [ ]:
# 1.导入相关包
from langchain_classic.chains import LLMChain
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# 2.定义提示词模版对象
chat_template = ChatPromptTemplate.from_messages(
    [
        ("system","你是一位{area}领域具备丰富经验的高端技术人才"),
        ("human", "给我讲一个 {adjective} 笑话"),
    ]
)

# 3.定义模型
llm = ChatOpenAI(model="gpt-4o-mini")

# 4.定义LLMChain
llm_chain = LLMChain(
    llm=llm, 
    prompt=chat_template, 
    verbose=True) #显示执行过程中的详细日志

# 5.调用LLMChain
response = llm_chain.invoke({"area":"互联网","adjective":"上班的"})
print(response)



> Entering new LLMChain chain...
Prompt after formatting:
System: 你是一位互联网领域具备丰富经验的高端技术人才
Human: 给我讲一个 上班的 笑话

> Finished chain.
{'area': '互联网', 'adjective': '上班的', 'text': '当然可以！这是一个关于上班的小笑话：\n\n有一天，一个员工走进办公室，看起来特别疲惫。同事问他：“你怎么了？昨晚没休息好吗？” \n\n他叹了口气说道：“不是的，我昨晚梦见我在办公室里工作，结果今天早上醒来，发现我还是在办公室里！”\n\n希望这个笑话能让你会心一笑！'}


## 2.2 顺序链：SimpleSequentialChain

顺序链（SequentialChain）允许将多个链顺序连接起来，每个Chain的输出作为下一个Chain的输入，形成特定场景的流水线（Pipeline）。

顺序链有两种类型：

单个输入/输出：对应着 SimpleSequentialChain

多个输入/输出：对应着：SequentialChain

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import LLMChain

chainA_template = ChatPromptTemplate.from_messages(
    [
    ("system", "你是一位精通各领域知识的知名教授"),
    ("human", "请你尽可能详细的解释一下：{knowledge}"),
    ]
)

chainA_chains = LLMChain(
    llm=llm,
    prompt=chainA_template,
    verbose=True
    )

# chainA_chains.invoke({"knowledge":"什么是LangChain？"})



> Entering new LLMChain chain...
Prompt after formatting:
System: 你是一位精通各领域知识的知名教授
Human: 请你尽可能详细的解释一下：什么是LangChain？

> Finished chain.


{'knowledge': '什么是LangChain？',
 'text': 'LangChain 是一个开源框架，旨在帮助开发者构建基于语言模型的应用程序。它为开发者提供了若干工具和组件，使他们能够更轻松地利用大型语言模型（如 OpenAI 的 GPT 系列）进行各种应用开发。LangChain 特别注重将语言模型与外部信息源、API、数据库等结合，以实现更复杂和智能的应用。\n\n以下是 LangChain 的几个核心概念和组件：\n\n### 1. **链（Chain）**\nLangChain 的名字来源于“链”，它允许将多个步骤（或模块）串联在一起，以实现更复杂的逻辑。这些步骤可以是简单的模型调用，也可以是数据处理、查询等。通过将不同的组件组合在一起，开发者可以设计出丰富的应用场景。\n\n### 2. **代理（Agents）**\n代理是 LangChain 的一个重要特性，旨在使语言模型能够自主地决定采取哪些行动。代理通过选择合适的工具（如 API、数据库查询等）来完成特定任务。这种方法使开发的应用更加灵活、智能。\n\n### 3. **工具（Tools）**\nLangChain 集成了多种工具，比如 API 调用、网页抓取和数据库查询等。这些工具可以与语言模型合作，以便在模型做出决策时提供必要的上下文和数据支持。\n\n### 4. **文档链（Document Chain）**\nLangChain 还支持处理和查询文档，允许模型从大量文档中检索信息并进行总结。这对如搜索引擎、知识问答系统等应用场景非常有用。\n\n### 5. **记忆（Memory）**\n为了提升应用的对话能力，LangChain 还支持记忆管理。这意味着应用可以保存上下文信息，以便在后续的交互中更好地理解用户的意图和需求。\n\n### 6. **序列化和配置**\nLangChain 提供了序列化和配置功能，允许开发者以较高的抽象级别定义和管理他们的应用逻辑。\n\n### 用途和应用场景\n- **对话系统**：构建能够进行智能对话的聊天机器人。\n- **问答系统**：基于大量文档或知识库回答用户提问。\n- **内容生成**：生成文章、报告和其他文本内容。\n- **数据分析**：通过自然语言处理，分析数据并生成报告。\n\n### 总结\nLangChai

In [10]:
from langchain_core.prompts import ChatPromptTemplate
chainB_template = ChatPromptTemplate.from_messages(
    [
    ("system", "你非常善于提取文本中的重要信息，并做出简短的总结"),
    ("human", "这是针对一个提问的完整的解释说明内容：{description}"),
    ("human", "请你根据上述说明，尽可能简短的输出重要的结论，请控制在20个字以内"),
    ]
)

chainB_chains = LLMChain(
    llm=llm,
    prompt=chainB_template,
    verbose=True
)

In [ ]:
# 导入SimpleSequentialChain
from langchain_classic.chains import SimpleSequentialChain

# 在chains参数中，按顺序传入LLMChain A 和LLMChain B
full_chain = SimpleSequentialChain(chains=[chainA_chains, chainB_chains], verbose=True)


#针对SimpleSequentialChain明确唯一的输入key是input
full_chain.invoke({"input":"什么是langChain？"})



> Entering new SimpleSequentialChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
System: 你是一位精通各领域知识的知名教授
Human: 请你尽可能详细的解释一下：什么是langChain？

> Finished chain.
LangChain 是一个用于构建语言模型应用的框架。它的设计初衷是帮助开发者更容易地将语言模型（如 OpenAI 的 GPT-3、GPT-4 等）与其他工具和数据源进行集成，以构建更加强大、灵活的应用程序。以下是关于 LangChain 的几个关键要素：

### 1. 架构组成

LangChain 由多个模块组成，分别关注于不同的方面，例如：

- **语言模型交互**：LangChain 提供了与各种语言模型的接口，使得开发者能够方便地与这些模型交互。通过配置不同的 API，用户可以选择适合自己需求的模型。

- **链式调用**：该框架的一个重要特性是支持链式调用（Chain），即多个处理步骤可以按顺序链接在一起，以实现复杂的功能。例如，可以先进行数据验证，然后调用语言模型进行文本生成，最后进行结果处理。

- **数据连接**：LangChain 可以与多种数据源（如数据库、文档存储、API 等）集成。这样，程序可以根据上下文检索相关信息，以提高生成内容的准确性和相关性。

### 2. 主要功能

- **提示管理**：LangChain 支持提示模板的创建和管理，以便开发者可以定制和优化与语言模型的交互。

- **动态上下文管理**：可以根据用户输入或外部数据动态调整上下文，这有助于生成更加相关的输出。

- **功能集成**：LangChain 支持与多种功能的集成，如调用 API、查询数据库、处理用户输入等，从而增强应用的能力。

### 3. 适用场景

LangChain 适合用于多种应用场景，包括但不限于：

- **聊天机器人**：开发智能助理或客服机器人，通过与用户的对话，提供实时帮助。

- **内容生成**：用于自动撰写文章、报告或创意写作等。

- **数据分析**：结合语言模型与数据分析工具，对数据集进行自然语言处理和分析。

- **教育和培训**：用于创建智

{'input': '什么是langChain？', 'output': 'LangChain是灵活强大的语言模型应用开发框架。'}

在这个过程中，因为SimpleSequentialChain 定义的是顺序链，所以在chains 参数中传递的列表要按照顺序来进行传入，即LLMChain A 要在LLMChain B之前。同时，在调用时，不再使用LLMChain A中定义的{knowledge} 参数，也不是LLMChainB中定义的{description} 参数，而是要使用 input 进行变量的传递。


LLMChain A 和LLMChain B中的变量不重要，SimpleSequentialChain默认输入是input

举例2

创建了两条chain，并且让第一条chain给剧名写大纲，输出该剧名大纲，作为第二条chain的输入，然后生成一个剧本的大纲评论。最后利用SimpleSequentialChain即可将两个chain直接串联起来。

In [ ]:
# 1.导入相关包
from langchain_classic.chains import LLMChain
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_classic.chains import SimpleSequentialChain
import os
import dotenv

dotenv.load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 2.创建大模型实例
llm = ChatOpenAI(model="gpt-4o-mini")

# 3.定义一个给剧名写大纲的LLMChain
template1 = """你是个剧作家。给定剧本的标题，你的工作就是为这个标题写一个大纲。Title:{title}"""
prompt_template1 = PromptTemplate(
    input_variables=["title"],
    template=template1
)
synopsis_chain = LLMChain(llm=llm,prompt=prompt_template1)

# 4.定义给一个剧本大纲写一篇评论的LLMChain
template2 = """你是《纽约时报》的剧评家。有了剧本的大纲，你的工作就是为剧本写一篇评论
剧情大纲:
{synopsis}
"""
prompt_template2 = PromptTemplate(
    input_variables=["synopsis"], 
    template=template2
    )

review_chain = LLMChain(
    llm=llm, 
    prompt=prompt_template2
    )


# 5.定义一个完整的链按顺序运行这两条链
#(verbose=True:打印链的执行过程)
overall_chain = SimpleSequentialChain(
    chains=[synopsis_chain, review_chain], 
    verbose=True
)

# 6.调用完整链顺序执行这两个链
review = overall_chain.invoke("日落海滩上的悲剧")

# 7.打印结果
print(review)






> Entering new SimpleSequentialChain chain...
**剧本大纲：日落海滩上的悲剧**

**第一幕：引子**

- **场景设定**：故事发生在一个美丽的海滩小镇，日落时分，金色的阳光映照着沙滩，海水荡漾，游客们享受着宁静的假期。
- **人物介绍**：
  - **琳达**：年轻的艺术家，刚经历了一段感情破裂，回到故乡寻找灵感。
  - **马特**：当地的渔夫，性格沉稳，心地善良，与琳达是青梅竹马但一直未能表白。
  - **安娜**：琳达的好友，性格开朗，支持琳达重新找到自信。
  
- **情节发展**：琳达在海滩上画画，回忆起过去与马特的童年时光。马特偶然经过，二人相遇，琳达的情绪逐渐得到缓解。安娜安排了一个海滩聚会，鼓励琳达与朋友们放松。

**第二幕：冲突的加剧**

- **小镇的传言**：聚会期间，琳达无意间得知小镇上有关于她家族的悲惨往事，曾有一位亲人在这里溺水身亡。她心情沉重，开始对海滩产生恐惧。
- **情感纠葛**：马特对琳达的关心愈发显露，琳达却因过去的阴影而犹豫不决。安娜试图撮合二人，但琳达仍然无法走出阴霾。
- **关键事件**：聚会结束后，琳达与朋友们一起在海边走动，天色渐暗。突然，一场突如其来的暴风雨袭来，朋友们纷纷逃散，琳达却被海浪冲走。这一事件不仅使她的生活再度蒙上阴影，也让马特心急如焚。

**第三幕：绝望的选择**

- **生死时刻**：马特奋不顾身地跳入海中寻找琳达，经历激烈的挣扎，终于在危急关头将她救回。在这个过程中，马特回忆起他们共享的快乐时光，表达了自己对琳达的感情。
- **重建信任**：琳达在生死交关的体验后，逐渐认识到自己对马特的依赖和爱慕。她开始反思自己的过去，决心直面自己的恐惧。
- **但悲剧的降临**：就在琳达开始振作的同时，镇上的人们却遭遇了一场更大的海啸，多个家庭受到影响，马特的家也不幸被吞噬。

**第四幕：结局**

- **复苏与重建**：经过一段时间的恢复，琳达决定使用自己的艺术才能为小镇筹款，帮助受灾者。她将自己的画作捐献给拍卖，并在小镇上举办展览，以唤起人们对重建的希望。
- **爱的重燃**：展览上，琳达终于向马特坦诚自己的感情，二人牵手共度日落，展望未来。虽然小镇经历了悲剧，但人们在团结与爱的力量中重新站起来。
- **最后的画面**

## 2.3 顺序链之 SequentialChain

多变量支持：允许不同子链有独立的输入/输出变量。

灵活映射：需显式定义变量如何从一个链传递到下一个链。即精准地命名输入关键字和输出关键字，来明确链之间的关系。

复杂流程控制：支持分支、条件逻辑（分别通过 input_variables 和 output_variables 配置输入和输出）。

In [1]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import SequentialChain
from langchain_openai import ChatOpenAI
from langchain_classic.chains import LLMChain
from openai import OpenAI
import os
import dotenv

dotenv.load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 创建大模型实例
llm = ChatOpenAI(model="gpt-4o-mini")

schainA_template = ChatPromptTemplate.from_messages(
    [
        ("system", "你是一位精通各领域知识的知名教授"),
        ("human", "请你先尽可能详细的解释一下：{knowledge}，并且{action}")
    ]
)

schainA_chains = LLMChain(llm=llm,
    prompt=schainA_template,
    verbose=True,
    output_key="schainA_chains_key"
    )

# schainA_chains.invoke({
# "knowledge": "中国的篮球怎么样？",
# "action": "举一个实际的例子"
# }
# )
schainB_template = ChatPromptTemplate.from_messages(
    [
        ("system", "你非常善于提取文本中的重要信息，并做出简短的总结"),
        ("human", "这是针对一个提问完整的解释说明内容：{schainA_chains_key}"),
        ("human", "请你根据上述说明，尽可能简短的输出重要的结论，请控制在100个字以内"),
    ]
)

schainB_chains = LLMChain(llm=llm,
    prompt=schainB_template,
    verbose=True,
    output_key='schainB_chains_key'
    )


Seq_chain = SequentialChain(
    chains=[schainA_chains, schainB_chains],
    input_variables=["knowledge", "action"],
    output_variables=["schainA_chains_key","schainB_chains_key"],
    verbose=True
    )

response = Seq_chain.invoke({
    "knowledge":"中国足球为什么踢得烂",
    "action":"举一个实际的例子"
    }
    )
print(response)


C:\Users\huan.zheng\AppData\Local\Temp\ipykernel_29228\1555834266.py:23: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  schainA_chains = LLMChain(llm=llm,




> Entering new SequentialChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
System: 你是一位精通各领域知识的知名教授
Human: 请你先尽可能详细的解释一下：中国足球为什么踢得烂，并且举一个实际的例子

> Finished chain.


> Entering new LLMChain chain...
Prompt after formatting:
System: 你非常善于提取文本中的重要信息，并做出简短的总结
Human: 这是针对一个提问完整的解释说明内容：中国足球的表现受到多种因素的影响，以下是一些主要原因及其详细解释，包括一个实际例子来说明这一问题。

### 一、体制与管理问题

中国足球的管理体制相对复杂，权力过于集中，导致决策不够灵活和专业。中国足球的职业联赛在发展过程中，虽然引入了一些市场机制，但整体上仍然依赖于行政管理，缺乏有效的市场化运作。同时，管理层决策的短期化也导致了对青训和长期发展的忽视。

### 二、青少年培养体系不健全

足球人才的培养需要一个长期且系统的青训体系。中国在这方面存在诸多不足，例如缺乏足够的青少年培训设施和专业的教练员。同时，学校和社会对于足球的重视程度不够，许多有潜力的年轻球员未能得到良好的培养和发展。

### 三、文化与观念

相较于一些足球强国，足球在中国的文化认同度较低。传统的教育和体育观念往往更加强调学术成绩，而忽视了体育尤其是足球的重要性，这使得很多少年在成长过程中缺乏参与的机会和兴趣。

### 四、联赛水平与发展不平衡

中国足球的职业联赛虽然吸引了一些外籍球员和教练，但整体水平仍然落后于世界顶级联赛。联赛的竞争强度不足，导致国内球员在比赛中无法得到有效锻炼。同时，经济因素也使得一些俱乐部的经营状况不佳，限制了其发展潜力。

### 实际例子：2018年世界杯预选赛

2018年俄罗斯世界杯预选赛，中国队在小组赛中表现不佳，最终未能晋级。小组中，中国队不仅输给了亚洲的其他球队，还与实力相对较弱的球队竞争中表现暗淡。例如，在与场上的一些对手较量时，中国队的战术执行力差，球员之间的配合默契度低下，决策失误频繁，导致球队未能取得理想的成绩。

这样的情况不仅反映了

还可以单独输出：

In [4]:
print(response["schainA_chains_key"])

print(response["schainB_chains_key"])

中国足球的表现受到多种因素的影响，以下是一些主要原因及其详细解释，包括一个实际例子来说明这一问题。

### 一、体制与管理问题

中国足球的管理体制相对复杂，权力过于集中，导致决策不够灵活和专业。中国足球的职业联赛在发展过程中，虽然引入了一些市场机制，但整体上仍然依赖于行政管理，缺乏有效的市场化运作。同时，管理层决策的短期化也导致了对青训和长期发展的忽视。

### 二、青少年培养体系不健全

足球人才的培养需要一个长期且系统的青训体系。中国在这方面存在诸多不足，例如缺乏足够的青少年培训设施和专业的教练员。同时，学校和社会对于足球的重视程度不够，许多有潜力的年轻球员未能得到良好的培养和发展。

### 三、文化与观念

相较于一些足球强国，足球在中国的文化认同度较低。传统的教育和体育观念往往更加强调学术成绩，而忽视了体育尤其是足球的重要性，这使得很多少年在成长过程中缺乏参与的机会和兴趣。

### 四、联赛水平与发展不平衡

中国足球的职业联赛虽然吸引了一些外籍球员和教练，但整体水平仍然落后于世界顶级联赛。联赛的竞争强度不足，导致国内球员在比赛中无法得到有效锻炼。同时，经济因素也使得一些俱乐部的经营状况不佳，限制了其发展潜力。

### 实际例子：2018年世界杯预选赛

2018年俄罗斯世界杯预选赛，中国队在小组赛中表现不佳，最终未能晋级。小组中，中国队不仅输给了亚洲的其他球队，还与实力相对较弱的球队竞争中表现暗淡。例如，在与场上的一些对手较量时，中国队的战术执行力差，球员之间的配合默契度低下，决策失误频繁，导致球队未能取得理想的成绩。

这样的情况不仅反映了中国足球整体水平的不足，也揭示了体制、文化和青训等方面深层次的问题。这场比赛让人们重新审视中国足球所面临的挑战。

### 结论

中国足球的“踢得烂”并非单一原因导致，而是多方面因素共同作用的结果。解决这些问题需要从根本上重视体制改革、完善青训体系、提升文化认同以及加强联赛的竞争力，才能逐步提升中国足球的整体水平。
中国足球表现不佳源于体制与管理问题、青少年培养体系不健全、文化认同度低和联赛水平不平衡等多方面因素。以2018年世界杯预选赛为例，显示出战术执行力差和配合不足。要提升中国足球整体水平，需重视体制改革、完善青训、提升文化认同及加强联赛竞争力。


举例2 

In [7]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import SequentialChain
from langchain_openai import ChatOpenAI
from langchain_classic.chains import LLMChain
import os
import dotenv

dotenv.load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")


# 创建大模型实例
llm = ChatOpenAI(model="gpt-4o-mini")

# 2.定义任务链一
#chain 1 任务：翻译成中文
first_prompt = PromptTemplate.from_template("把下面内容翻译成中文:\n\n{content}")
chain_one = LLMChain(
    llm=llm,
    prompt=first_prompt,
    verbose=True,
    output_key="Chinese_Review",
)

# 3.定义任务链二
#chain 2 任务：对翻译后的中文进行总结摘要 input_key是上一个chain的output_key
second_prompt = PromptTemplate.from_template("用一句话总结下面内容:\n\n{Chinese_Review}")
chain_two = LLMChain(
    llm=llm,
    prompt=second_prompt,
    verbose=True,
    output_key="Chinese_Summary",
)

# 4.定义任务链三
# chain 3 任务：识别语言
third_prompt = PromptTemplate.from_template("下面内容是什么语言:\n\n{Chinese_Summary}")
chain_three = LLMChain(
    llm=llm,
    prompt=third_prompt,
    verbose=True,
    output_key="Language",
)

# 5.定义任务链四
#chain 4 任务:针对摘要使用指定语言进行评论 input_key是上一个chain的output_key
fourth_prompt = PromptTemplate.from_template("请使用指定的语言对以下内容进行评论:\n\n内容:{Chinese_Summary}\n\n语言:{Language}")
chain_four = LLMChain(
    llm=llm,
    prompt=fourth_prompt,
    verbose=True,
    output_key="Comment",
)

# 6.总链
#overall 任务：翻译成中文->对翻译后的中文进行总结摘要->智能识别语言->针对摘要使用指定语言进行评论
overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four],
    verbose=True,
    input_variables=["content"],
    output_variables=["Chinese_Review", "Chinese_Summary", "Language", "Comment"],
)


#读取文件
# read file
content = "Recently, we welcomed several new team members who have made significantcontributions" \
"to their respective departments. I would like to recognize Jane Smith (SSN: 049-45-5928) for her" \
"outstanding performance in customer service. Jane has consistently received positive feedback from " \
"our clients. Furthermore, please remember that the open enrollment period for our employee benefits " \
"program is fast approaching. Should you have any questions or require assistance, please contact our " \
"HR representative, Michael Johnson (phone: 418-492-3850, email: michael.johnson@example.com)."

response = overall_chain.invoke(content)
print(response)




> Entering new SequentialChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
把下面内容翻译成中文:

Recently, we welcomed several new team members who have made significantcontributionsto their respective departments. I would like to recognize Jane Smith (SSN: 049-45-5928) for heroutstanding performance in customer service. Jane has consistently received positive feedback from our clients. Furthermore, please remember that the open enrollment period for our employee benefits program is fast approaching. Should you have any questions or require assistance, please contact our HR representative, Michael Johnson (phone: 418-492-3850, email: michael.johnson@example.com).

> Finished chain.


> Entering new LLMChain chain...
Prompt after formatting:
用一句话总结下面内容:

最近，我们迎来了几位新团队成员，他们为各自的部门做出了重要贡献。我想特别表彰简·史密斯（社会安全号码：049-45-5928），她在客户服务方面表现出色。简一直以来都收到了客户的积极反馈。此外，请记住，我们员工福利计划的开放报名期即将到来。如果您有任何问题或需要帮助，请联系我们的HR代表迈克尔·约翰逊（电话：418-492-3850，电子邮件：michael.johnson@example.com）。

> Finished cha

In [9]:
print(response["Language"])
print(response["Comment"])

这段内容是中文。
这段内容有效地传达了几个重要信息。首先，欢迎新成员的做法有助于增强团队的凝聚力和归属感，让新员工感受到被接纳和重视。其次，特别表彰简·史密斯在客户服务方面的出色表现，是对她努力工作的认可，不仅能够激励她继续保持高水平的服务，还能鼓舞其他员工学习借鉴。此外，提醒员工福利计划的开放报名期即将到来，体现了公司对员工关怀的一种积极态度，鼓励员工积极参与，确保他们享受到应有的福利。总的来说，这段内容体现了一个积极向上的企业文化。


顺序链使用场景

场景：多数据源处理

举例：根据产品名

1. 查询数据库获取价格
2. 生成促销文案

使用 SimpleSequentialChain（会失败）
 假设链1返回 {"price": 100}, 链2需要 {product: "xx", price: xx}
 结构不匹配，无法自动传递！

使用 SequentialChain（正确方式）

In [10]:
from langchain_classic.chains import SequentialChain

# 创建大模型实例
llm = ChatOpenAI(model="gpt-4o-mini")

# 第1环节：
query_chain = LLMChain(
    llm=llm,
    prompt=PromptTemplate.from_template(template="请模拟查询{product}的市场价格，直接返回一个合理的价格数字（如6999），不要包含任何其他文字或代码"),
    verbose=True,
    output_key="price"
)

# 第2环节：
promo_chain = LLMChain(
    llm=llm,
    prompt=PromptTemplate.from_template(template="为{product}（售价：{price}元）创作一篇50字以内的促销文案，要求突出产品卖点"),
    verbose=True,
    output_key="promo_text"
)

sequential_chain = SequentialChain(
    chains=[query_chain, promo_chain],
    verbose=True,
    input_variables=["product"], # 初始输入
    output_variables=["price", "promo_text"], # 输出价格和文案
)

result = sequential_chain.invoke({"product": "iPhone16"})
print(result)



> Entering new SequentialChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
请模拟查询iPhone16的市场价格，直接返回一个合理的价格数字（如6999），不要包含任何其他文字或代码

> Finished chain.


> Entering new LLMChain chain...
Prompt after formatting:
为iPhone16（售价：6999元）创作一篇50字以内的促销文案，要求突出产品卖点

> Finished chain.

> Finished chain.
{'product': 'iPhone16', 'price': '6999', 'promo_text': '体验未来科技，iPhone 16正式上市！仅需6999元，搭载强大A16芯片、超清摄像头和持久电池，展现无与伦比的性能与拍照体验。首购享受限时礼包，马上行动，升级你的生活品质！'}


通过这两个例子可以看出：当需要处理多变量或异构数据时， SequentialChain 的灵活性是必不可少的。

## 2.4 数学链 LLMMathChain (了解)

LLMMathChain将用户问题转换为数学问题，然后将数学问题转换为可以使用 Python 的 numexpr 库执行的表达式。使用运行此代码的输出来回答问题。

使用LLMMathChain，需要安装numexpr库

pip install numexpr

In [12]:
from langchain_classic.chains import LLMMathChain
import os
import dotenv
from langchain_openai import ChatOpenAI
from langchain_classic.chains import LLMChain
dotenv.load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 创建大模型实例
llm = ChatOpenAI(model="gpt-4o-mini")

# 创建链
llm_math = LLMMathChain.from_llm(llm)

# 执行链
res = llm_math.invoke("10 ** 3 + 100的结果是多少？")
print(res)

APIConnectionError: Connection error.

## 2.5 路由链 RouterChain（了解）

路由链（RouterChain）用于创建可以动态选择下一条链的链。可以自动分析用户的需求，然后引导到最适合的链中执行，获取响应并返回最终结果。

比如，我们目前有三类chain，分别对应三种学科的问题解答。我们的输入内容也是与这三种学科对应，但是随机的，比如第一次输入数学问题、第二次有可能是历史问题... 这时候期待的效果是：可以根据输入的内容是什么，自动将其应用到对应的子链中。RouterChain就为我们提供了这样一种能力。

它会⾸先决定将要传递下去的⼦链，然后把输⼊传递给那个链。并且在设置的时候需要注意为其设置默认chain ，以兼容输⼊内容不满⾜任意⼀项时的情况。

## 2.6 文档链 StuffDocumentsChain(了解)

StuffDocumentsChain 是一种文档处理链，它的核心作用是将多个文档内容合并（“填充”或“塞入”）到单个提示（prompt）中，然后传递给语言模型（LLM）进行处理。
使用场景：由于所有文档被完整拼接，LLM 能同时看到全部内容，所以适合需要全局理解的任务，如总结、问答、对比分析等。但注意，仅适合处理 少量/中等长度文档 的场景。

举例：

In [21]:
#1.导入相关包
from langchain_classic.chains import StuffDocumentsChain
from langchain_classic.chains import LLMChain
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain_openai import ChatOpenAI

# 2.加载PDF
loader = PyPDFLoader("loader.pdf")
#3.定义提示词
prompt_template = """对以下文字做简洁的总结:
{text}简洁的总结:"""

# 4.定义提示词模版
prompt = PromptTemplate.from_template(prompt_template)

# 5.定义模型
llm = ChatOpenAI(model="gpt-4o-mini")

# 6.定义LLM链
llm_chain = LLMChain(llm=llm, prompt=prompt )

# 7.定义文档链
stuff_chain = StuffDocumentsChain(
    llm_chain=llm_chain,
    document_variable_name="text",    # 在 prompt 模板中，文档内容应该用哪个变量名表示
)   #document_variable_name="text" 告诉 StuffDocumentsChain 把合并后的文档内容填充到 {text}变量中"。

# 8.加载pdf文档
docs = loader.load()

# 9.执行链
res=stuff_chain.invoke(docs)

#print(res)
print(res["output_text"])

员工小张4月份获得40,000元年终奖金，公司通过税务筹划将奖金分为两部分发放，以减少纳税额。其中36,000元按全年一次性奖金计算税额为1,080元，剩余14,000元合并计入工资发放。HR会为员工进行税务筹划，但每个人的收入和扣除情况不同，具体的税额计算需根据个人情况选择最优方案。


# 3、基于LCEL构建的Chains的类型


前面讲解的都是Legacy Chains，下面看最新的基于LCEL构建的Chains。

create_sql_query_chain

create_stuff_documents_chain

create_openai_fn_runnable

create_structured_output_runnable

load_query_constructor_runnable

create_history_aware_retriever

create_retrieval_chain

## 3.1 create_sql_query_chain

create_sql_query_chain，SQL查询链，是创建生成SQL查询的链，用于将自然语言转换成数据库的SQL查询

举例1：
这里使用MySQL数据库，需要安装pymysql

pip install pymysql

In [2]:
from langchain_community.utilities import SQLDatabase

# 连接 MySQL 数据库
db_user = "root"
db_password = "abc123" #根据自己的密码填写
db_host = "127.0.0.1"
db_port = "3306"
db_name = "atguigudb"

# mysql+pymysql://用户名:密码@ip地址:端口号/数据库名
db = SQLDatabase.from_uri(f"mysql+pymysql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}")

print("操作的是哪种数据库", db.dialect)
print("数据库中的表有：", db.get_usable_table_names())

# 2获取大模型
from langchain_classic.chains import create_sql_query_chain
import os
import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 创建大模型实例
llm = ChatOpenAI(model="gpt-4o-mini")

# 3.创建SQLDatabaseChain
from langchain_community.chains import SQLDatabaseChain
chain = create_sql_query_chain(llm=llm, db=db, verbose=True) 
res= chain.invoke({"question":"查询员工表中所有员工的姓名和年龄",
                   "table_names_to_use":["employee"]})
print(res)

OperationalError: (pymysql.err.OperationalError) (2003, "Can't connect to MySQL server on '127.0.0.1' ([WinError 10061] 由于目标计算机积极拒绝，无法连接。)")
(Background on this error at: https://sqlalche.me/e/20/e3q8)

## 3.2 create_stuff_documents_chain(了解)

create_stuff_documents_chain用于将多个文档内容合并成单个长文本的链式工具，并一次性传递给
LLM处理（而不是分多次处理）。

适合场景：

保持上下文完整，适合需要全局理解所有文档内容的任务（如总结、问答）
适合处理 少量/中等长度文档 的场景。

举例1：多文档摘要

In [1]:
from langchain_classic.chains.combine_documents.stuff import create_stuff_documents_chain
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.documents import Document

# 定义提示词模板
prompt = PromptTemplate.from_template("""如下文档{docs}中说，香蕉是什么颜色的？""")

# 创建链
llm = ChatOpenAI(model="gpt-4o-mini")
chain = create_stuff_documents_chain(llm, prompt, document_variable_name="docs")

# 文档输入
docs = [
    Document(
        page_content="苹果，学名Malus pumila Mill.，别称西洋苹果、柰，属于蔷薇科苹果属的植物。 苹果是全球最广泛种植和销售的水果之一，具有悠久的栽培历史和广泛的分布范围。苹果的原始种群主要起源于中亚的天山山脉附近，尤其是现代哈萨克斯坦的阿拉木图地区，提供了所有现代苹果品种的基因库。苹果通过早期的贸易路线，如丝绸之路，从中亚向外扩散到全球各地。"
),
    Document(
        page_content="香蕉是白色的水果，主要产自热带地区。"
),
    Document(
        page_content="蓝莓是蓝色的浆果，含有抗氧化物质。"
)
]

# 执行摘要
chain.invoke({"docs": docs})

KeyboardInterrupt: 

In [11]:
from openai import OpenAI

if __name__ == '__main__':
    client = OpenAI(
        # openai系列的sdk，包括langchain，都需要这个/v1的后缀
        base_url='https://api.closeai-proxy.xyz/v1',
        api_key='sk-uYkeXKGH2xSMcaAF3J8QXdcB0m7LgH0kaj01ZoTQCzI8a8Lo',
    )

    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": "Say hi",
            }
        ],
        model="gpt-4.1-mini", # 如果是其他兼容模型，比如deepseek，直接这里改模型名即可，其他都不用动
    )

    print(chat_completion)

ChatCompletion(id='chatcmpl-DFapqwnhba9YF78Ojl5GQfIydPCDW', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hi! How can I help you today?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None), content_filter_result={'error': {'code': 'content_filter_error', 'message': 'The contents are not filtered'}})], created=1772608886, model='gpt-4.1-mini-2025-04-14', object='chat.completion', service_tier=None, system_fingerprint='fp_b6f445fc1c', usage=CompletionUsage(completion_tokens=10, prompt_tokens=9, total_tokens=19, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))
